# שיעור 20 — הכנת נתונים: חסרים, X/y ואימון/בדיקה
יחידה 3 | 2026–2027 | מעבר מחקר נתונים להכנת מודל.

ידע קודם: DataFrame, חסרים, ממוצע ובחירת עמודות. עיבוד שלנו לפי סיכומי מפגשים 7 ו־9 בהשתלמות, לא מחברת מקור של המרצה.
[CSV](https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv) | [מקור Palmer Penguins](https://github.com/allisonhorst/palmerpenguins) | [קטלוג](https://github.com/samiabo75/Data-Science-ML-Course/blob/main/Datasets/README.md).
ייחוס: Horst, Hill & Gorman (2020), palmerpenguins, DOI 10.5281/zenodo.3960218; גרסת הקורס 344×7. פרטי שימוש במקור. שמרו עותק אישי.

## פתיחה — איך נבדוק חיזוי על דוגמאות שלא שימשו ללמידה?
המשימה העתידית היא סיווג species לפי אורך ועומק המקור. אם נראה למודל גם את התשובות של כל הדוגמאות שעליהן נבחן אותו, לא נקבל בדיקת הכללה הוגנת. לכן מפרידים מראש אימון ובדיקה.

אימון משמש ללימוד פרמטרים. בדיקה נשמרת להערכה בסוף; אינה משמשת לבחירת K או שיטת השלמה שוב ושוב. בהמשך נשתמש באימות בתוך האימון לבחירת חלופות.

## 1. בודקים חסרים לפני החלטה

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
print("ממדי המקור:", df.shape)
print("חסרים לפי עמודה:")
print(df.isna().sum())

isna מחזירה בדיקת חסר לכל תא; sum סופרת True בכל עמודה. כאן 2 חסרים בכל אחת מארבע המדידות ו־11 ב־sex. אין חסר ב־species.

חסר אינו אפס. לא מוחקים 11 רשומות רק מפני ש־sex חסרה אם איננו משתמשים בה במשימה. קודם בוחרים שאלה ומאפיינים, ואז מחליטים מה רלוונטי.

## 2. מגדירים מאפיינים ויעד
X תהיה טבלה של שתי מדידות מספריות; y תהיה סדרת תוויות species. זהו סיווג כי היעד קטגוריאלי. אין צורך להמיר כאן תוויות טקסט למספרים לקראת KNN של scikit-learn.

לא מנחשים תווית יעד חסרה לצורך אימון מונחה. אם יש רשומות ללא יעד ידוע, מחריגים אותן מהמשימה המפוקחת ומתעדים; הן עשויות לשמש בעתיד כחלק נפרד ללא תוויות.

In [ ]:
model_df = df.dropna(subset=["species"]).copy()
feature_columns = ["bill_length_mm", "bill_depth_mm"]
X = model_df[feature_columns]
y = model_df["species"]
print("שורות לפני ואחרי דרישת יעד ידוע:", len(df), len(model_df))
print("ממדי X:", X.shape)
print("ממדי y:", y.shape)
print("חסרים ב-X:")
print(X.isna().sum())

בקובץ הנוכחי שום רשומה לא הוסרה בגלל יעד חסר. subset מציינת בדיוק אילו עמודות לבדוק; copy יוצרת טבלת עבודה נפרדת. species אינה בתוך X — הכנסת התשובה עצמה לקלט היא דליפת מידע.

## 3. מחלקים לפני שלומדים ערכי השלמה
train_test_split מחלקת את X ואת y יחד, כדי לשמור התאמת רשומה ותגית. היא מחזירה ארבעה אובייקטים; זו השמה של ארבע תוצאות, לא הגדרת ארבע פונקציות.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("X אימון:", X_train.shape, "y אימון:", y_train.shape)
print("X בדיקה:", X_test.shape, "y בדיקה:", y_test.shape)
print("חסרים באימון:")
print(X_train.isna().sum())
print("חסרים בבדיקה:")
print(X_test.isna().sum())

test_size=0.25 מקצה כרבע לבדיקה; random_state משחזר חלוקה נתונה, לא משפר דיוק; stratify=y שומר בקירוב יחס מחלקות בחלוקת סיווג זו. בגרסה שנבדקה 258 לאימון ו־86 לבדיקה.

זו דוגמה לחלוקה אקראית של תצפיות מתאימות. בנתוני זמן, קבוצות או מדידות חוזרות מאותו נבדק יידרש תכנון אחר; אין לערבב אוטומטית כל מאגר.

## 4. מחשבים ממוצעי השלמה מאימון בלבד
כדי להתמקד במנגנון נבחר כרגע ממוצע לכל מאפיין מספרי. אין טענה שזו שיטה מיטבית לכל מאגר. חציון או מחיקה מנומקת הם חלופות, שאותן בוחרים בהמשך באמצעות אימות ולא לפי ציון הבדיקה.

In [ ]:
training_means = X_train.mean()
print("ממוצעי האימון להשלמה:")
print(training_means)
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)
print("חסרים לאחר השלמה באימון:")
print(X_train_ready.isna().sum())
print("חסרים לאחר השלמה בבדיקה:")
print(X_test_ready.isna().sum())

X_train.mean מחזירה ממוצע לכל עמודה מספרית, תוך דילוג על חסרים. fillna מתאימה לפי שמות העמודות. אותם ערכים בדיוק משמשים לשתי הקבוצות. לא מחשבים X_test.mean לצורך השלמה ולא משלימים לפי X.mean לפני החלוקה.

הטבלאות החדשות שומרות מספר שורות, עמודות ואינדקסים. המקור ונתוני החלוקה הגולמיים נשמרו. אם עמודת אימון כולה חסרה, אין ממוצע זמין; אין להמשיך כאילו הבעיה נפתרה.

## 5. דוגמה ידנית לדליפה — נתוני הוראה בלבד
באימון ערכים 10,20 וחסר; בבדיקה 1000 וחסר. ממוצע האימון 15. השלמה תקינה משתמשת ב־15 בשתי הקבוצות. ממוצע כל הידועים יחד הוא (10+20+1000)/3≈343.33 — ערך שלמד מן הבדיקה. שינוי כזה אינו הוכחה שהציון תמיד יעלה, אלא שהבדיקה השפיעה על תהליך הלמידה.

In [ ]:
toy_train = pd.DataFrame({"value": [10, 20, None]})
toy_test = pd.DataFrame({"value": [1000, None]})
toy_means = toy_train.mean()
print("ממוצע אימון:", toy_means)
print("אימון לאחר השלמה:")
print(toy_train.fillna(toy_means))
print("בדיקה לאחר אותה השלמה:")
print(toy_test.fillna(toy_means))

## תוצר לפרויקט — יומן הכנת נתונים
תעדו יעד ומאפיינים, חסרים רלוונטיים, כלל לטיפול ביעד חסר, חלוקה וזרע אקראי, החלטת השלמה ופרמטרים שנלמדו מאימון, ממדים לפני/אחרי ומגבלה. ההשלמה יוצרת ערכים משוערים, אינה מגלה את המדידה האמיתית ועלולה לצמצם פיזור.

## כרטיס יציאה
1. מה שונה בין חסר במאפיין לחסר בתווית יעד?
2. מדוע מחלקים X ו־y יחד?
3. מאיזו קבוצה לומדים את ממוצעי ההשלמה?
4. האם random_state הוא פרמטר שמשפר דיוק?
5. האם מותר לבחור שיטת טיפול לפי ניסויים חוזרים על Test?

אין אימון מודל היום. השלב הבא יהיה הכנת קנה מידה לקראת KNN. Penguins להוראה, לא מאגר גמר מאושר.

In [ ]:
print("ממדי המקור נשמרו:", df.shape)
print("חסרי המקור נשמרו:")
print(df.isna().sum())